In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Load the persisted data from Notebook 1
df_clean = pd.read_parquet('group2_stocks.parquet')
df_benchmark = pd.read_parquet('benchmark.parquet')

print(f"✅ Data loaded successfully. stocks: {df_clean.shape}, benchmark: {df_benchmark.shape}")

### VOLATILITY TERM STRUCTURE

In [ ]:
# --- PHASE 2: VOLATILITY PROFILING ---

# 1. Base Calculation: Daily Log Returns
# Volatility is the standard deviation of these returns.
log_returns = np.log(df_clean / df_clean.shift(1)).fillna(0)

# 2. Define Horizons (Roadmap Step 2)
horizons = {
    'Weekly_Vol': 5,     # 5 trading days
    'Monthly_Vol': 21,    # 21 trading days
    'Quarterly_Vol': 63   # 63 trading days
}

# Container for the "Feature Matrix" (One row per stock, used for Step 3 Clustering)
vol_profile_median = pd.DataFrame(index=df_clean.columns)

# Container for "Full Time Series" (Used for Step 2 Visualization)
vol_time_series = {}

print("🚀 Calculating Annualized Realized Volatility...")

for name, window in horizons.items():
    # A. Calculate Rolling Standard Deviation
    # B. Annualize by multiplying by the square root of 252 trading days
    rolling_vol = log_returns.rolling(window=window).std() * np.sqrt(252)
    
    # Store the full time-series for plotting regimes
    vol_time_series[name] = rolling_vol
    
    # Store the median value to represent the stock's "Typical Risk Signature"
    # We use median to ensure one-off market spikes don't skew our cluster features
    vol_profile_median[name] = rolling_vol.median()

# Clean up any initial NaN values caused by the rolling window offset
vol_profile_median = vol_profile_median.fillna(0)

print(f"✅ Volatility Profiling Complete.")
print(f"📊 Feature Matrix Shape: {vol_profile_median.shape} (Stocks x Vol-Horizons)")

In [ ]:
# --- CELL 2: VOLATILITY TERM STRUCTURE ---
# Prepare data for Plotly
df_reset = vol_profile_median.reset_index().rename(columns={'index': 'Ticker'})
df_long = df_reset.melt(id_vars='Ticker', var_name='Horizon', value_name='Volatility')

fig = px.line(
    df_long, x='Horizon', y='Volatility', color='Ticker', markers=True,
    title='Volatility Term Structure (Median)',
    hover_data={'Ticker': True, 'Volatility': ':.3f'}
)
fig.update_layout(template="plotly_white", height=600)
fig.show()

### Volatility Time Series

In [ ]:
# --- CELL 3: VOLATILITY TIME SERIES ---
for window_name, df_vol in vol_time_series.items():
    # Drop NaNs for plotting
    df_clean_vol = df_vol.dropna().reset_index()
    date_col = df_clean_vol.columns[0]
    
    # Melt
    df_long_vol = df_clean_vol.melt(id_vars=date_col, var_name='Ticker', value_name='Annualized Volatility')
    
    # Plot
    fig = px.line(
        df_long_vol, x=date_col, y='Annualized Volatility', color='Ticker',
        title=f'{window_name}: Time Series History',
        hover_data={'Ticker': True, 'Annualized Volatility': ':.3f'}
    )
    fig.update_layout(template="plotly_white", height=500)
    fig.show()

### Volatility Leaderboard

In [ ]:
# --- CELL 4: VOLATILITY LEADERBOARD ---
# Purpose: Rank stocks by their Monthly (21-day) Volatility to identify the 'Risk Leaders'.
top_10_vol = vol_profile_median.sort_values(by='Monthly_Vol', ascending=False).head(10)

fig_lead = px.bar(
    top_10_vol, x=top_10_vol.index, y='Monthly_Vol',
    title="Top 10 High-Volatility Stocks (Monthly Horizon)",
    labels={'index': 'Ticker', 'Monthly_Vol': 'Annualized Vol (%)'},
    template="plotly_white", color='Monthly_Vol', color_continuous_scale='Reds'
)
fig_lead.show()